# 과제 B — 같은 일을 세 가지로 엮고 숫자로 비교하기

보안 공지 한 건(1번 Log4Shell)으로 **보안 공지 카드** 한 장을 만듭니다. 전문가 셋이 나눠 씁니다.

```
             +-- 요약가       --> [요약]
공지 본문 ---+-- 영향도 평가자 --> [영향도]   ==> 카드 한 장
             +-- 조치 작성자   --> [조치]
```

| 방식 | 흐름을 정하는 주체 | 참고 |
|---|---|---|
| ① 코드로 엮기 | 우리 파이썬 (`asyncio.gather`) | `openai_agent_code` Step 3~7 |
| ② 에이전트를 도구로 | 관리자 LLM (`as_tool`) | Step 8 |
| ③ 핸드오프 | 관리자 LLM (`handoffs`) | Step 9 |

바꾸는 변수는 **엮는 방식 하나**입니다. 공지·모델·전문가 지시문은 세 방식이 모두 같습니다.

## 0. 준비

In [1]:
import asyncio
import json
import time

import pandas as pd
from dotenv import load_dotenv
from agents import Agent, Runner, trace

load_dotenv(override=True)   # OPENAI_API_KEY 하나만 씁니다

MODEL_NAME = "gpt-5.4-mini"
REPEATS = 3

with open("advisories.json", encoding="utf-8") as f:
    ADVISORIES = json.load(f)

# 비교는 한 건으로만 한다. 공지를 섞으면 방식 차이인지 공지 차이인지 알 수 없다.
ADV = next(a for a in ADVISORIES if a["no"] == 1)

ADVISORY_TEXT = f"""CVE: {ADV['cve_id']}
제품: {ADV['label']}

공지 본문:
{ADV['advisory']}"""

print("모델:", MODEL_NAME, "/ 반복:", REPEATS)
print(f"공지: {ADV['no']}번 {ADV['cve_id']} ({ADV['label']}) — 본문 {len(ADV['advisory'])}자")

모델: gpt-5.4-mini / 반복: 3
공지: 1번 CVE-2021-44228 (Apache Log4j2) — 본문 2500자


## 1. 전문가 세 명 (필수 1)

- 공통 배경 `intro` 를 변수로 빼서 세 지시문에 더합니다. 한 곳만 고치면 셋 다 바뀝니다
- 세 전문가는 **같은 모델**을 씁니다
- 공통 배경에 **"공지에 없는 것을 지어내지 말 것"** 을 넣었습니다

카드 모양(`CARD_FORMAT`)도 변수로 뺐습니다. ① 은 파이썬이, ②·③ 은 LLM 이 이 모양으로 조립합니다.

In [2]:
intro = """당신은 보안 관제팀의 취약점 분석가입니다.
보안 공지 한 건을 받아 사내 공유용 '보안 공지 카드'의 한 부분을 씁니다.
반드시 공지에 적힌 내용만 근거로 쓰고, 공지에 없는 버전·수치·사례를 지어내지 마세요.
공지에서 확인할 수 없는 것은 '공지에 명시되지 않음'이라고 쓰세요.
한국어로, 머리말이나 인사 없이 본론만 쓰세요.
"""

summarizer = Agent(
    name="요약가",
    instructions=intro + "당신의 담당은 '요약'입니다. 무엇이 어떻게 취약한지 3문장 이내로 요약하세요.",
    model=MODEL_NAME,
)
impact_rater = Agent(
    name="영향도 평가자",
    instructions=intro + "당신의 담당은 '영향도'입니다. 영향받는 버전, 공격 조건, 악용 시 결과를 "
                         "근거로 위험도를 긴급/높음/보통/낮음 중 하나로 매기고 3문장 이내로 이유를 쓰세요.",
    model=MODEL_NAME,
)
action_writer = Agent(
    name="조치 작성자",
    instructions=intro + "당신의 담당은 '조치'입니다. 담당자가 바로 실행할 조치를 우선순위 순으로 "
                         "3개 이내의 글머리표로 쓰세요. 패치 버전과 임시 완화책을 구분하세요.",
    model=MODEL_NAME,
)

EXPERTS = [summarizer, impact_rater, action_writer]

CARD_FORMAT = """[보안 공지 카드] <CVE 번호> — <제품>
■ 요약
<요약>
■ 영향도
<영향도>
■ 조치
<조치>"""

for a in EXPERTS:
    print(f"{a.name:10} 모델={a.model}")

요약가        모델=gpt-5.4-mini
영향도 평가자    모델=gpt-5.4-mini
조치 작성자     모델=gpt-5.4-mini


## 2. 재는 함수 (필수 5)

`result` 안에 사용량을 담는 곳이 **두 군데** 있습니다.

| 위치 | 담긴 것 |
|---|---|
| `result.raw_responses[i].usage` | 이 `Runner.run` 이 **직접** 받은 모델 응답들. 핸드오프로 넘겨받은 에이전트의 응답도 포함 |
| `result.context_wrapper.usage` | 위 + **`as_tool()` 로 부른 전문가가 안에서 따로 부른 LLM 호출까지** |

SDK 소스(`agents/agent.py` 의 `as_tool`)를 보면, 전문가를 돌릴 때 `usage=context.usage` 로
**관리자의 사용량 객체를 그대로 넘겨** 같은 곳에 더해 갑니다.
반면 전문가의 모델 응답은 관리자의 `raw_responses` 에는 들어가지 않습니다.

그래서 `raw_responses` 로만 세면 ②·③ 은 **관리자 호출만** 세어져 실제보다 싸게 나옵니다.
아래 표에서는 **`context_wrapper.usage` 를 정답**으로 쓰고, `raw_responses` 값은 비교용 열(`raw_*`)로 함께 남깁니다.

In [3]:
def measure(result):
    """result 하나에서 두 군데 사용량을 모두 뽑는다."""
    u = result.context_wrapper.usage
    raw = result.raw_responses
    return {
        "llm_calls": u.requests,
        "in_tok": u.input_tokens,
        "out_tok": u.output_tokens,
        "raw_llm_calls": len(raw),
        "raw_in_tok": sum(r.usage.input_tokens for r in raw),
        "raw_out_tok": sum(r.usage.output_tokens for r in raw),
    }


def add_up(measures):
    """방식 ① 처럼 Runner.run 을 여러 번 했을 때 합친다."""
    return {k: sum(m[k] for m in measures) for k in measures[0]}


def tool_calls(result):
    """도구 호출 기록 (핸드오프 호출은 handoff_call_item 이라 여기 안 잡힌다)."""
    return [getattr(item.raw_item, "name", "?")
            for item in result.new_items if item.type == "tool_call_item"]

## 3. 방식 ① 코드로 엮기 (필수 2)

셋을 `asyncio.gather` 로 **동시에** 부르고, 카드는 **파이썬 문자열**로 조립합니다.
카드 조립에는 LLM 을 쓰지 않습니다.

In [4]:
def assemble_card(summary, impact, action):
    return (CARD_FORMAT
            .replace("<CVE 번호>", ADV["cve_id"])
            .replace("<제품>", ADV["label"])
            .replace("<요약>", summary.strip())
            .replace("<영향도>", impact.strip())
            .replace("<조치>", action.strip()))


async def run_code():
    t0 = time.time()
    with trace("과제B ① 코드로 엮기"):
        results = await asyncio.gather(*(Runner.run(a, ADVISORY_TEXT) for a in EXPERTS))
    card = assemble_card(*(r.final_output for r in results))
    sec = time.time() - t0

    return {
        "method": "① 코드",
        "expert_calls": len(results),     # gather 가 정확히 3번 부른다 (도구 호출이 아니므로 tool_call_item 없음)
        "calls": [a.name for a in EXPERTS],
        **add_up([measure(r) for r in results]),
        "sec": sec,
        "card_len": len(card),
        "handoff": None,
        "last_agent": None,
        "card": card,
    }

## 4. 방식 ② 에이전트를 도구로 (필수 3)

`tool_name` 은 영문(`^[a-zA-Z0-9_-]+$`)만 됩니다. 누가 하는 일인지는 `tool_description` 에 한글로 적었습니다.

관리자 지시문에는 **역할만** 적고, 이번 일은 `task` 에 짧게 적었습니다.
절차를 단계별로 다 써 두면 ② 가 ① 을 말로 옮긴 것이 되기 때문입니다.
`gather` 가 보장하던 "셋 다 받는다"는 여기서는 **"세 결과가 모두 준비될 때까지"** 한 줄로 부탁할 뿐입니다.

In [5]:
expert_tools = [
    summarizer.as_tool(
        tool_name="summarizer",
        tool_description="요약가. 보안 공지를 받아 무엇이 어떻게 취약한지 요약한다."),
    impact_rater.as_tool(
        tool_name="impact_rater",
        tool_description="영향도 평가자. 보안 공지를 받아 위험도(긴급/높음/보통/낮음)와 그 근거를 쓴다."),
    action_writer.as_tool(
        tool_name="action_writer",
        tool_description="조치 작성자. 보안 공지를 받아 담당자가 실행할 조치를 쓴다."),
]

tool_manager = Agent(
    name="카드 편집장",
    instructions=intro + "당신은 전문가 도구들이 쓴 결과를 모아 보안 공지 카드를 완성하는 편집장입니다.\n"
                         "카드 형식:\n" + CARD_FORMAT,
    tools=expert_tools,
    model=MODEL_NAME,
)

TOOL_TASK = f"""아래 공지로 보안 공지 카드 한 장을 만드세요.
요약·영향도·조치는 각 전문가 도구에게 맡기고, 세 결과가 모두 준비될 때까지 카드를 쓰지 마세요.

{ADVISORY_TEXT}"""

for t in expert_tools:
    print(f"- {t.name:14} {t.description}")


async def run_tool():
    t0 = time.time()
    with trace("과제B ② 에이전트를 도구로"):
        result = await Runner.run(tool_manager, TOOL_TASK)
    sec = time.time() - t0

    calls = tool_calls(result)
    return {
        "method": "② 도구",
        "expert_calls": len(calls),
        "calls": calls,
        **measure(result),
        "sec": sec,
        "card_len": len(result.final_output),
        "handoff": None,
        "last_agent": result.last_agent.name,
        "card": result.final_output,
    }

- summarizer     요약가. 보안 공지를 받아 무엇이 어떻게 취약한지 요약한다.
- impact_rater   영향도 평가자. 보안 공지를 받아 위험도(긴급/높음/보통/낮음)와 그 근거를 쓴다.
- action_writer  조치 작성자. 보안 공지를 받아 담당자가 실행할 조치를 쓴다.


## 5. 방식 ③ 핸드오프 (필수 4)

관리자는 전문가 도구로 재료만 모으고, **카드 조립은 `Card_Writer` 에게 넘깁니다.**
핸드오프 대상의 `name` 은 영문이어야 `transfer_to_Card_Writer` 도구가 제대로 만들어집니다.

관리자가 넘기지 않고 혼자 카드를 써도 결과물은 멀쩡한 카드입니다.
그래서 실행마다 **`handoff_output_item` 발생 여부**와 **`last_agent.name`** 을 둘 다 기록합니다.

In [6]:
card_writer = Agent(
    name="Card_Writer",
    instructions=intro + "당신은 앞에서 모은 요약·영향도·조치를 그대로 옮겨 보안 공지 카드 한 장으로 조립합니다.\n"
                         "카드 형식:\n" + CARD_FORMAT,
    model=MODEL_NAME,
)

handoff_manager = Agent(
    name="재료 수집 관리자",
    instructions=intro + "당신은 전문가 도구들로 카드 재료를 모으고, 카드 조립은 카드 작성자에게 맡기는 관리자입니다.",
    tools=expert_tools,
    handoffs=[card_writer],
    model=MODEL_NAME,
)

HANDOFF_TASK = f"""아래 공지로 보안 공지 카드 한 장을 만드세요.
요약·영향도·조치는 각 전문가 도구에게 맡기고, 세 결과가 모두 준비되면 카드 작성자에게 넘기세요.

{ADVISORY_TEXT}"""

print("도구", len(handoff_manager.tools), "개 / 핸드오프", [h.name for h in handoff_manager.handoffs])


async def run_handoff():
    t0 = time.time()
    with trace("과제B ③ 핸드오프"):
        result = await Runner.run(handoff_manager, HANDOFF_TASK)
    sec = time.time() - t0

    kinds = [item.type for item in result.new_items]
    calls = tool_calls(result)
    return {
        "method": "③ 핸드오프",
        "expert_calls": len(calls),
        "calls": calls,
        **measure(result),
        "sec": sec,
        "card_len": len(result.final_output),
        "handoff": "handoff_output_item" in kinds,
        "last_agent": result.last_agent.name,
        "card": result.final_output,
    }

도구 3 개 / 핸드오프 ['Card_Writer']


## 6. 세 방식 × REPEATS 회 실행 (필수 6)

방식을 한 번에 몰아서 돌리면 그 시간대 API 상태가 한 방식에만 몰립니다.
그래서 **회차 안에서 세 방식을 번갈아** 돌립니다 (①②③ ①②③ ...).

In [7]:
RUNNERS = [run_code, run_tool, run_handoff]

records = []
for i in range(1, REPEATS + 1):
    for fn in RUNNERS:
        rec = await fn()
        rec["run"] = i
        records.append(rec)
        print(f"[{i}회] {rec['method']:8} 전문가 {rec['expert_calls']}회 / LLM {rec['llm_calls']}회 / "
              f"토큰 {rec['in_tok']:>6}+{rec['out_tok']:<5} / {rec['sec']:5.1f}초 / 카드 {rec['card_len']}자")

[1회] ① 코드     전문가 3회 / LLM 3회 / 토큰   2390+513   /   2.9초 / 카드 1006자


[1회] ② 도구     전문가 3회 / LLM 5회 / 토큰   6550+2640  /  12.3초 / 카드 723자


[1회] ③ 핸드오프   전문가 3회 / LLM 8회 / 토큰  14670+2722  /  21.0초 / 카드 575자


[2회] ① 코드     전문가 3회 / LLM 3회 / 토큰   2390+501   /   2.4초 / 카드 1071자


[2회] ② 도구     전문가 3회 / LLM 6회 / 토큰   8793+2801  /  15.9초 / 카드 746자


[2회] ③ 핸드오프   전문가 3회 / LLM 6회 / 토큰   9710+2609  /  13.4초 / 카드 712자


[3회] ① 코드     전문가 3회 / LLM 3회 / 토큰   2390+477   /   3.0초 / 카드 956자


[3회] ② 도구     전문가 3회 / LLM 5회 / 토큰   4254+1478  /   8.8초 / 카드 749자


[3회] ③ 핸드오프   전문가 3회 / LLM 8회 / 토큰  14919+2817  /  21.0초 / 카드 603자


### 실행별 기록

`raw_*` 열은 `raw_responses` 로만 센 값입니다. ① 은 두 값이 같고, ②·③ 은 크게 차이 납니다.

In [8]:
df = pd.DataFrame(records)
cols = ["method", "run", "expert_calls", "llm_calls", "raw_llm_calls",
        "in_tok", "raw_in_tok", "out_tok", "raw_out_tok", "sec", "card_len", "handoff", "last_agent"]
df[cols].round({"sec": 1})

,method,run,expert_calls,llm_calls,raw_llm_calls,in_tok,raw_in_tok,out_tok,raw_out_tok,sec,card_len,handoff,last_agent
0,① 코드,1,3,3,3,2390,2390,513,513,2.9,1006,None,NaN
1,② 도구,1,3,5,2,6550,4415,2640,2056,12.3,723,None,카드 편집장
2,③ 핸드오프,1,3,8,5,14670,12280,2722,2260,21.0,575,True,Card_Writer
3,① 코드,2,3,3,3,2390,2390,501,501,2.4,1071,None,NaN
4,② 도구,2,3,6,3,8793,6417,2801,2315,15.9,746,None,카드 편집장
5,③ 핸드오프,2,3,6,3,9710,7542,2609,2075,13.4,712,True,Card_Writer
6,① 코드,3,3,3,3,2390,2390,477,477,3.0,956,None,NaN
7,② 도구,3,3,5,2,4254,3233,1478,965,8.8,749,None,카드 편집장
8,③ 핸드오프,3,3,8,5,14919,12529,2817,2280,21.0,603,True,Card_Writer


### 호출 순서와 핸드오프 확인

"전문가 3회"가 숫자로만 맞고 **같은 전문가를 두 번** 부른 것은 아닌지, 순서까지 봅니다.

In [9]:
for r in records:
    extra = ""
    if r["method"] == "③ 핸드오프":
        extra = f"  | 핸드오프 발생={r['handoff']}  마지막 에이전트={r['last_agent']}"
    print(f"[{r['run']}회] {r['method']:8} {' -> '.join(r['calls'])}{extra}")

[1회] ① 코드     요약가 -> 영향도 평가자 -> 조치 작성자
[1회] ② 도구     summarizer -> impact_rater -> action_writer
[1회] ③ 핸드오프   summarizer -> impact_rater -> action_writer  | 핸드오프 발생=True  마지막 에이전트=Card_Writer
[2회] ① 코드     요약가 -> 영향도 평가자 -> 조치 작성자
[2회] ② 도구     summarizer -> impact_rater -> action_writer
[2회] ③ 핸드오프   summarizer -> impact_rater -> action_writer  | 핸드오프 발생=True  마지막 에이전트=Card_Writer
[3회] ① 코드     요약가 -> 영향도 평가자 -> 조치 작성자
[3회] ② 도구     summarizer -> impact_rater -> action_writer
[3회] ③ 핸드오프   summarizer -> impact_rater -> action_writer  | 핸드오프 발생=True  마지막 에이전트=Card_Writer


## 7. 비교표 — 평균 (최소~최대)

In [10]:
METRICS = {
    "expert_calls": "전문가 호출",
    "llm_calls": "LLM 호출",
    "in_tok": "입력 토큰",
    "out_tok": "출력 토큰",
    "sec": "걸린 초",
    "card_len": "카드 글자 수",
    "raw_llm_calls": "(raw_responses만) LLM 호출",
    "raw_in_tok": "(raw_responses만) 입력 토큰",
}


def fmt(s, digits):
    return f"{s.mean():,.{digits}f} ({s.min():,.{digits}f}~{s.max():,.{digits}f})"


order = ["① 코드", "② 도구", "③ 핸드오프"]
table = pd.DataFrame({
    label: {m: fmt(df.loc[df.method == m, key], 1 if key == "sec" else 0) for m in order}
    for key, label in METRICS.items()
}).T
table

,① 코드,② 도구,③ 핸드오프
전문가 호출,3 (3~3),3 (3~3),3 (3~3)
LLM 호출,3 (3~3),5 (5~6),7 (6~8)
입력 토큰,"2,390 (2,390~2,390)","6,532 (4,254~8,793)","13,100 (9,710~14,919)"
출력 토큰,497 (477~513),"2,306 (1,478~2,801)","2,716 (2,609~2,817)"
걸린 초,2.7 (2.4~3.0),12.4 (8.8~15.9),18.5 (13.4~21.0)
카드 글자 수,"1,011 (956~1,071)",739 (723~749),630 (575~712)
(raw_responses만) LLM 호출,3 (3~3),2 (2~3),4 (3~5)
(raw_responses만) 입력 토큰,"2,390 (2,390~2,390)","4,688 (3,233~6,417)","10,784 (7,542~12,529)"


In [11]:
# 토큰 배율 (①을 1로 놓고), 정답 기준(context_wrapper.usage)
base = df[df.method == "① 코드"][["in_tok", "out_tok", "llm_calls", "sec"]].mean()
ratio = pd.DataFrame({m: df[df.method == m][["in_tok", "out_tok", "llm_calls", "sec"]].mean() / base
                      for m in order}).T
ratio["total_tok"] = [
    df[df.method == m][["in_tok", "out_tok"]].sum(axis=1).mean()
    / df[df.method == "① 코드"][["in_tok", "out_tok"]].sum(axis=1).mean()
    for m in order
]
ratio.round(2)

,in_tok,out_tok,llm_calls,sec,total_tok
① 코드,1.00,1.00,1.00,1.00,1.00
② 도구,2.73,4.64,1.78,4.50,3.06
③ 핸드오프,5.48,5.46,2.44,6.73,5.48


In [12]:
# 흔들림 비교: 방식마다 각 지표의 (최대-최소)/평균
spread = pd.DataFrame({
    m: {label: (lambda s: (s.max() - s.min()) / s.mean() if s.mean() else 0)(df.loc[df.method == m, key])
        for key, label in list(METRICS.items())[:6]}
    for m in order
})
(spread * 100).round(0).astype(int).astype(str) + "%"

,① 코드,② 도구,③ 핸드오프
전문가 호출,0%,0%,0%
LLM 호출,0%,19%,27%
입력 토큰,0%,69%,40%
출력 토큰,7%,57%,8%
걸린 초,21%,57%,41%
카드 글자 수,11%,4%,22%


## 8. 결과 카드 (각 방식 1회차)

In [13]:
for m in order:
    r = next(x for x in records if x["method"] == m and x["run"] == 1)
    print("=" * 70)
    print(m)
    print("=" * 70)
    print(r["card"])
    print()

① 코드
[보안 공지 카드] CVE-2021-44228 — Apache Log4j2
■ 요약
Log4j 2.16.0 미만 버전은 LDAP JNDI 파서를 통해 원격 코드 실행(RCE)에 취약하며, 공격자가 로그 메시지나 파라미터를 제어할 수 있으면 LDAP 등 공격자 제어 JNDI 엔드포인트를 통해 임의 코드 실행이 가능합니다. 영향을 받는 것은 직접적으로 `org.apache.logging.log4j:log4j-core`이며, `log4j-api`는 함께 같은 버전으로 맞춰야 합니다. 공지에 따르면 2.15.0 이전이 영향 범위이고, 2.16.0에서 JNDI는 기본 비활성화되며 메시지 룩업이 완전히 제거됩니다.
■ 영향도
긴급: 공지에 따르면 2.16.0 이전의 Apache Log4j2(특히 2.15.0 미만)가 영향받으며, 공격자가 로그 메시지나 파라미터를 제어할 수 있고 message lookup substitution이 활성화된 경우 원격 코드 실행(RCE)이 가능합니다. 신뢰되지 않거나 사용자 제어 데이터가 로그에 기록되는 상황에서 악용될 수 있어 영향이 매우 큽니다. 악용 시 결과는 애플리케이션에 대한 RCE이며, 세부 공격 경로는 공지에 명시되지 않음입니다.
■ 조치
- **패치 적용 우선**: `org.apache.logging.log4j:log4j-core`를 **2.16.0 이상**으로 업데이트하고, 사용 중이면 `log4j-api`도 **동일 버전**으로 맞추세요. 공지상 **2.15.0**은 초기 수정이지만 모든 상황에서 충분하지 않습니다.
- **임시 완화책**: 패치 전까지는 **신뢰할 수 없는/사용자 제어 입력을 로그에 남기지 않도록** 조치하고, 특히 예외 추적·인증 실패 등 **로그에 포함되는 외부 입력**을 점검하세요. (구체적 비활성화 설정은 공지에 명시되지 않음)
- **적용 범위 확인**: 영향 대상은 **`log4j-core`**이며, 현재 사용 버전이 **2.15.0 미만**인지 먼저 확인하세요. (세부 점검 방법은 공

## 9. 결론

> 아래 숫자는 위 3회 실행(방식마다 3회, 1번 Log4Shell 공지 한 건)에서 나온 값입니다. 다시 실행하면 값이 바뀌므로 이 결론도 그 출력 기준입니다.

### 1. 세 방식 중 어느 것을 쓰겠는가 — 그 근거는 어느 숫자인가

**① 코드로 엮기**를 쓰겠습니다.
토큰이 평균 2,887개(입력 2,390 + 출력 497)로 ②의 8,838개(3.06배), ③의 15,816개(5.48배)보다 적고, LLM 호출도 3회로 고정입니다(②는 5~6회, ③은 6~8회).
시간도 평균 2.7초로 ②(12.4초)·③(18.5초)보다 짧습니다. 다만 아래 4번에서 보듯 이 차이를 전부 "일이 적어서"로 볼 수는 없습니다.
무엇보다 ①은 **입력 토큰이 3회 모두 2,390으로 같았고**, ②·③은 입력 토큰이 40~69% 폭으로 흔들렸습니다. 순서가 정해진 일을 LLM에게 맡기면 비용을 미리 알 수 없게 됩니다.
품질 면에서도 ②·③이 나은 점은 없었습니다. ③의 1회차 카드는 오히려 **영향도 칸에서 등급(긴급/높음/보통/낮음)이 빠졌습니다.** Card_Writer가 다시 쓰면서 핵심 칸을 잃은 것입니다.

### 2. "전문가 3회 호출"이 지켜졌는가 — 지켜졌다면 대신 어디가 흔들렸는가

**지켜졌습니다.** 9회 실행 모두 전문가 호출이 정확히 3회였고, 순서도 매번 summarizer → impact_rater → action_writer였습니다.
③은 3회 모두 `handoff_output_item`이 발생했고 `last_agent.name`도 `Card_Writer`였습니다.
대신 **비용과 시간이 흔들렸습니다.** LLM 호출 수가 ②는 5~6회, ③은 6~8회로 달라졌고, 입력 토큰은 ②가 4,254~8,793(약 2배 차이), ③이 9,710~14,919로 벌어졌습니다. 시간도 ②가 8.8~15.9초, ③이 13.4~21.0초였습니다.
**전문가에게 넘어간 공지도 흔들렸습니다.** 전문가 쪽 입력(`in_tok − raw_in_tok`)은 ①에서는 항상 2,390이지만, ②에서는 2,135 / 2,376 / **1,021**, ③에서는 2,390 / 2,168 / 2,390이었습니다. 관리자가 공지를 그대로 넘기지 않고 **줄여서** 넘긴 회차가 있다는 뜻입니다. 같은 "전문가 3회"라도 전문가가 본 자료가 달랐습니다.
그 결과 카드 길이도 ③이 575~712자(22%)로 흔들렸고, 위에서 말한 등급 누락도 생겼습니다.

### 3. 토큰 차이는 몇 배였는가 — 그 차이는 어디서 생기는가

①을 1로 놓으면 총 토큰은 ② **3.06배**, ③ **5.48배**입니다(입력 2.73/5.48배, 출력 4.64/5.46배).
차이는 **관리자(와 Card_Writer) 쪽**에서 생깁니다. 전문가 쪽 입력은 위에서 보듯 ① 이하였는데도 총량은 3~5배가 됐습니다.
관리자 비용은 두 가지입니다. (a) 관리자가 턴마다 공지 전문과 그때까지 받은 도구 결과를 **다시 읽고**, (b) 공지를 도구 인자로 **써서** 넘깁니다. 출력 배율이 4.6~5.5배로 큰 이유가 (b)입니다. ③은 관리자 턴이 더 많고(raw 호출 3~5회) Card_Writer가 지금까지의 대화 전체를 이어받아 읽기 때문에 ②보다 더 큽니다.
**측정 위치가 중요했습니다.** `raw_responses`로만 세면 ②의 LLM 호출이 평균 2회로 **①(3회)보다 적게** 나옵니다. 힌트가 말한 "있을 수 없는 숫자"입니다. 전문가가 `as_tool` 안에서 부른 LLM 호출은 `raw_responses`에는 안 남고, 관리자와 공유하는 `context_wrapper.usage`에만 더해지기 때문입니다.

### 4. 속도 비교에서 공정하지 않은 부분은 무엇인가

①은 `asyncio.gather`로 전문가 셋을 **동시에** 불렀습니다. ②·③은 관리자가 턴을 **차례로** 진행하며 전문가를 부르고, 카드 조립에도 LLM 호출이 더 듭니다.
그래서 ①이 4.5배(②)·6.7배(③) 빠른 데는 **"일이 적어서"(LLM 호출 3회 대 5~8회)와 "동시에 해서"가 섞여 있고, 이 실험만으로는 둘을 나눌 수 없습니다.** ①을 순차로 돌린 시간이나 전문가별 소요 시간은 재지 않았습니다.
반대 방향의 섞임도 있습니다. 관리자가 한 응답에서 도구 여러 개를 요청하면 SDK가 그것들을 동시에 실행하는데, 관리자가 셋을 한 턴에 불렀는지 나눠 불렀는지는 기록하지 않았습니다. ②의 raw 호출이 2회인 회차는 셋을 한 번에 부른 것으로 보이지만, 확인하지는 않았습니다.
그 밖에도 반복이 3회뿐이라 최소~최대 범위가 실제 분포를 다 보여 준다고 할 수 없습니다. ②·③의 흔들림 폭(40~69%)을 보면 3회로는 평균도 꽤 움직일 수 있습니다.
